# TP3 - Regression and model generalization

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor

data = pd.DataFrame({
    "Student": [f"S{i}" for i in range(1, 11)],
    "Study Hours": np.arange(1, 11),
    "Exam Score": [42, 48, 55, 59, 66, 70, 78, 81, 89, 91],
})
X = data[["Study Hours"]]
y = data["Exam Score"]
data

## Part A - Hypothesis spaces

In [ ]:
print("Dimensions:", data.shape)
display(data.describe())

plt.figure(figsize=(7, 4))
plt.scatter(X, y, color="black", label="observations")
plt.xlabel("Study hours")
plt.ylabel("Exam score")
plt.title("Study hours and exam scores")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Model comparison

In [ ]:
models_a = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression (degree 2)": Pipeline([
        ("polynomial", PolynomialFeatures(degree=2)),
        ("regression", LinearRegression()),
    ]),
    "Polynomial Regression (degree 3)": Pipeline([
        ("polynomial", PolynomialFeatures(degree=3)),
        ("regression", LinearRegression()),
    ]),
    "Decision Tree (depth 1)": DecisionTreeRegressor(max_depth=1, random_state=0),
    "Decision Tree (depth 3)": DecisionTreeRegressor(max_depth=3, random_state=0),
    "KNN (k=3)": KNeighborsRegressor(n_neighbors=3),
}

results_a = []
x_grid = pd.DataFrame({"Study Hours": np.linspace(X["Study Hours"].min(), X["Study Hours"].max(), 300)})
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color="black", s=45, label="observations", zorder=3)

for name, model in models_a.items():
    model.fit(X, y)
    train_prediction = model.predict(X)
    results_a.append({
        "Model": name,
        "Training RMSE": mean_squared_error(y, train_prediction) ** 0.5,
        "Training R²": r2_score(y, train_prediction),
    })
    plt.plot(x_grid, model.predict(x_grid), label=name)

results_a = pd.DataFrame(results_a).sort_values("Training RMSE")
display(results_a.round(4))
plt.xlabel("Study hours")
plt.ylabel("Predicted exam score")
plt.title("Training fits: different hypothesis spaces")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Part B - Complexity and generalization

In [ ]:
loo = LeaveOneOut()
loo_rows = []
for iteration, (train_index, test_index) in enumerate(loo.split(X), start=1):
    loo_rows.append({
        "Iteration": iteration,
        "Training students": ", ".join(data.loc[train_index, "Student"]),
        "Test student": data.loc[test_index[0], "Student"],
        "Test hours": X.iloc[test_index[0], 0],
    })
display(pd.DataFrame(loo_rows))

def loocv_rmse(model):
    """Return the square root of the mean LOOCV squared error."""
    scores = cross_val_score(
        model, X, y, cv=LeaveOneOut(), scoring="neg_mean_squared_error"
    )
    return np.sqrt(-scores.mean())

degree_rows = []
degree_models = {}
for degree in [1, 2, 3, 5, 9]:
    model = Pipeline([
        ("polynomial", PolynomialFeatures(degree=degree)),
        ("regression", LinearRegression()),
    ])
    model.fit(X, y)
    training_rmse = mean_squared_error(y, model.predict(X)) ** 0.5
    degree_rows.append({
        "Polynomial Degree": degree,
        "Training RMSE": training_rmse,
        "LOOCV RMSE": loocv_rmse(model),
    })
    degree_models[degree] = model

degree_results = pd.DataFrame(degree_rows)
display(degree_results.round(4))

plt.figure(figsize=(7, 4))
plt.plot(degree_results["Polynomial Degree"], degree_results["LOOCV RMSE"], marker="o")
plt.xlabel("Polynomial degree")
plt.ylabel("LOOCV RMSE")
plt.title("Generalization error versus polynomial complexity")
plt.xticks([1, 2, 3, 5, 9])
plt.grid(alpha=0.3)
plt.show()
best_degree = int(degree_results.loc[degree_results["LOOCV RMSE"].idxmin(), "Polynomial Degree"])
print("Lowest LOOCV RMSE polynomial degree:", best_degree)

## Part C - Features and regularization

In [ ]:
def polynomial_pipeline(degree, estimator):
    return Pipeline([
        ("polynomial", PolynomialFeatures(degree=degree)),
        ("scaler", StandardScaler()),
        ("regression", estimator),
    ])

feature_rows = []
for degree in [1, 3, 5]:
    model = polynomial_pipeline(degree, LinearRegression())
    feature_rows.append({
        "Representation": f"x to degree {degree}",
        "LOOCV RMSE": loocv_rmse(model),
    })
feature_results = pd.DataFrame(feature_rows)
display(feature_results.round(4))

regularization_rows = []
alpha_values = [0.01, 0.1, 1, 10]
for model_name, estimator_class in [("Ridge", Ridge), ("Lasso", Lasso)]:
    for alpha in alpha_values:
        model = polynomial_pipeline(5, estimator_class(alpha=alpha, max_iter=100000))
        regularization_rows.append({
            "Model": model_name,
            "Feature degree": 5,
            "Alpha": alpha,
            "LOOCV RMSE": loocv_rmse(model),
        })
regularization_results = pd.DataFrame(regularization_rows)
display(regularization_results.round(4))

best_ridge_alpha = regularization_results.query("Model == 'Ridge'").sort_values("LOOCV RMSE").iloc[0]["Alpha"]
best_lasso_alpha = regularization_results.query("Model == 'Lasso'").sort_values("LOOCV RMSE").iloc[0]["Alpha"]
print("Best Ridge alpha:", best_ridge_alpha)
print("Best Lasso alpha:", best_lasso_alpha)

## Part D - Final comparison

In [ ]:
final_models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression": polynomial_pipeline(best_degree, LinearRegression()),
    "Ridge Regression": polynomial_pipeline(5, Ridge(alpha=best_ridge_alpha)),
    "Lasso Regression": polynomial_pipeline(5, Lasso(alpha=best_lasso_alpha, max_iter=100000)),
    "Decision Tree": DecisionTreeRegressor(max_depth=3, random_state=0),
    "KNN": KNeighborsRegressor(n_neighbors=3),
}

final_rows = []
for name, model in final_models.items():
    model.fit(X, y)
    predictions = model.predict(X)
    final_rows.append({
        "Model": name,
        "Configuration": {
            "Polynomial Regression": f"degree={best_degree}",
            "Ridge Regression": f"degree=5, alpha={best_ridge_alpha}",
            "Lasso Regression": f"degree=5, alpha={best_lasso_alpha}",
            "Decision Tree": "max_depth=3",
            "KNN": "k=3",
        }.get(name, "default"),
        "Training RMSE": mean_squared_error(y, predictions) ** 0.5,
        "LOOCV RMSE": loocv_rmse(model),
    })

final_results = pd.DataFrame(final_rows).sort_values("LOOCV RMSE")
display(final_results.round(4))

plt.figure(figsize=(10, 6))
plt.scatter(X, y, color="black", s=45, label="observations", zorder=3)
for name, model in final_models.items():
    grid_prediction = model.predict(x_grid)
    plt.plot(x_grid, grid_prediction, label=name)
plt.xlabel("Study hours")
plt.ylabel("Exam score")
plt.title("Final model comparison")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print("Best final model by LOOCV RMSE:", final_results.iloc[0]["Model"])
print("Best final LOOCV RMSE:", round(final_results.iloc[0]["LOOCV RMSE"], 4))